# Silver OrderItems

Run top to bottom in the Retail Lakehouse (WSL) kernel. Source: Bronze order_items. Output: Silver order_items.

DQ004-006 stop the build after error evidence is saved. Line-total tolerance of PKR 0.02 is provisional.

Rejections are saved in ignored error/data/order_items with their original JSON record. Run the numbered notebooks in order.

This notebook also derives estimated_line_cost = quantity × product cost_price and estimated_line_margin = line_total − estimated_line_cost. Margin here is before order-level discounts and tax. Rerun all cells after this change.


In [ ]:
import os  # Configure Java before starting Spark.
import shutil  # Locate the Linux Java executable.
from pathlib import Path  # Build table paths.
from uuid import uuid4  # Tag rejected rows from this run.
from pyspark.sql import SparkSession, functions as F  # Use Spark DataFrames and expressions.
from pyspark.sql.window import Window  # Count duplicate business keys.
from delta import configure_spark_with_delta_pip  # Load Delta Lake support.
java = shutil.which("java")  # Find Java inside WSL.
if java is None:  # Stop if Java is absent.
    raise RuntimeError("Java is unavailable in WSL")  # Explain the startup failure.
os.environ["JAVA_HOME"] = str(Path(java).resolve().parent.parent)  # Avoid the Windows Java path.
project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Point to this project.
if not project_root.is_dir():  # Confirm the WSL mount exists.
    raise FileNotFoundError(project_root)  # Do not run against the wrong folder.
builder = (SparkSession.builder.master("local[2]")  # Keep local resource use modest.
    .appName("RetailSilverOrderItems")  # Label this table's Spark job.
    .config("spark.sql.shuffle.partitions", "4")  # Use fewer shuffle tasks for small local tables.
    .config("spark.ui.enabled", "false")  # Avoid a second Spark UI port.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta SQL.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))  # Register Delta tables.
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start or reuse Spark.
print("Spark version:", spark.version)  # Confirm startup.


In [ ]:
table_name = 'order_items'  # Name the Silver output table.
bronze_name = 'order_items'  # Name its Bronze source table.
business_key = ['order_item_id']  # Columns that uniquely identify one row.
fields = [  # Define cleaning and type conversion for each source column.
    ('order_item_id', 'text'),  # Convert order_item_id to text.
    ('order_id', 'text'),  # Convert order_id to text.
    ('product_id', 'text'),  # Convert product_id to text.
    ('quantity', 'int'),  # Convert quantity to int.
    ('unit_price', 'decimal(12,2)'),  # Convert unit_price to decimal(12,2).
    ('line_total', 'decimal(12,2)'),  # Convert line_total to decimal(12,2).
    ('currency_code', 'text'),  # Convert currency_code to text.
]
references = [  # Define Silver tables needed for foreign-key checks.
    ('orders', 'order_id', 'order_id', 'order_found', False),  # Match order_id to orders.order_id.
    ('products', 'product_id', 'product_id', 'product_found', False),  # Match product_id to products.product_id.
]
rejection_rules = [  # Rows matching these rules go to error evidence.
    ('missing_order_item_id', "order_item_id IS NULL OR order_item_id = ''"),  # Check missing_order_item_id.
    ('duplicate_order_item_id', 'id_occurrences > 1'),  # Check duplicate_order_item_id.
    ('unknown_order', "order_id IS NULL OR order_id = '' OR order_found = false"),  # Check unknown_order.
    ('unknown_product', "product_id IS NULL OR product_id = '' OR product_found = false"),  # Check unknown_product.
    ('invalid_quantity', 'quantity IS NULL OR quantity <= 0'),  # Check invalid_quantity.
    ('invalid_unit_price', 'unit_price IS NULL OR unit_price < 0'),  # Check invalid_unit_price.
    ('invalid_line_total', 'line_total IS NULL OR line_total < 0'),  # Check invalid_line_total.
    ('missing_estimated_line_cost', 'estimated_line_cost IS NULL'),  # Require a usable product cost and quantity.
    ('missing_estimated_line_margin', 'estimated_line_margin IS NULL'),  # Require a derived margin.
    ('invalid_currency', "currency_code IS NULL OR currency_code <> 'PKR'"),  # Check invalid_currency.
]
critical_rules = [  # These rejection rules also stop the pipeline.
    ('unknown_order', "order_id IS NULL OR order_id = '' OR order_found = false"),  # Critical: unknown_order.
    ('unknown_product', "product_id IS NULL OR product_id = '' OR product_found = false"),  # Critical: unknown_product.
    ('invalid_quantity', 'quantity IS NULL OR quantity <= 0'),  # Critical: invalid_quantity.
    ('invalid_unit_price', 'unit_price IS NULL OR unit_price < 0'),  # Critical: invalid_unit_price.
    ('missing_estimated_line_cost', 'estimated_line_cost IS NULL'),  # Critical: missing product cost.
    ('missing_estimated_line_margin', 'estimated_line_margin IS NULL'),  # Critical: missing derived margin.
]
warning_rules = [  # These issues are reported but do not reject a row.
    ('line_total_formula_variance', 'abs(line_total - quantity * unit_price) > 0.02'),  # Warning: line_total_formula_variance.
]


In [ ]:
bronze_path = project_root / "lakehouse" / "bronze" / "data" / bronze_name  # Locate Bronze Delta.
bronze_df = spark.read.format("delta").load(str(bronze_path))  # Read its current committed version.
expected_columns = [name for name, kind in fields]  # Build the source schema contract.
if bronze_df.columns != expected_columns:  # Detect missing, extra, or reordered fields.
    raise ValueError(f"Unexpected Bronze columns: {bronze_df.columns}")  # Stop on schema drift.
bronze_count = bronze_df.count()  # Count every source record.
print("Bronze rows:", bronze_count)  # Show the input count.
bronze_df.printSchema()  # Show original Spark types.
bronze_df.show(3, truncate=False)  # Preview three records.


In [ ]:
def cleaned_column(name, kind):  # Convert one field according to the dictionary.
    value = F.col(name)  # Select the original Bronze field.
    if kind == "text":  # Required text stays text.
        return F.trim(value)  # Remove surrounding whitespace.
    if kind == "optional_text":  # Optional text can be absent.
        text = F.trim(value)  # Trim the value first.
        return F.when(text == "", F.lit(None)).otherwise(text)  # Make blank text null.
    target_type = "int" if kind == "optional_int" else kind  # Use Spark's integer type.
    return value.try_cast(target_type)  # Convert safely; invalid input becomes null.
clean_df = bronze_df.select(  # Keep one output row per Bronze row.
    *[cleaned_column(name, kind).alias(name) for name, kind in fields],  # Clean business fields.
    F.to_json(F.struct(*[F.col(name) for name in expected_columns])).alias("source_record"),  # Preserve original values.
)
if clean_df.count() != bronze_count:  # Conversion must never lose a row.
    raise ValueError("Cleaning changed row count")  # Stop if reconciliation fails.
clean_df.printSchema()  # Show converted Silver candidate types.


In [ ]:
checked_base_df = clean_df  # Start with every cleaned source row.
for table, source_key, target_key, marker, allow_distinct in references:  # Check each foreign key.
    path = project_root / "lakehouse" / "silver" / "data" / table  # Locate the reference table.
    lookup_df = spark.read.format("delta").load(str(path)).select(target_key)  # Read just its key.
    if allow_distinct:  # A product category occurs on many products.
        lookup_df = lookup_df.distinct()  # Keep one lookup row per category.
    elif lookup_df.count() != lookup_df.distinct().count():  # Primary keys must be unique.
        raise ValueError(f"Duplicate reference keys in {table}")  # Avoid multiplying rows.
    lookup_df = lookup_df.withColumn(marker, F.lit(True))  # Mark successful matches.
    checked_base_df = checked_base_df.join(  # Preserve all source rows.
        lookup_df, checked_base_df[source_key] == lookup_df[target_key], "left"  # Match to Silver.
    ).drop(lookup_df[target_key]).fillna(False, subset=[marker])  # Remove the extra key.
    if checked_base_df.count() != bronze_count:  # A lookup must preserve source row count.
        raise ValueError(f"Reference join changed row count: {table}")  # Stop on a bad join.
print("Rows after lookups:", checked_base_df.count())  # Confirm all input rows remain.
product_cost_path = project_root / "lakehouse" / "silver" / "data" / "products"  # Find typed product costs.
product_cost_df = spark.read.format("delta").load(str(product_cost_path)).select(  # Read one cost per product.
    "product_id", F.col("cost_price").alias("_product_cost_price")  # Rename cost as a temporary helper.
)
if product_cost_df.count() != product_cost_df.select("product_id").distinct().count():  # Require one cost per product.
    raise ValueError("Silver products contain duplicate IDs")  # Prevent multiplied order lines.
checked_base_df = checked_base_df.join(product_cost_df, on="product_id", how="left")  # Add cost without dropping lines.
if checked_base_df.count() != bronze_count:  # Ensure the cost join kept the order-item grain.
    raise ValueError("Product cost join changed order-item row count")  # Stop on an unsafe join.
checked_base_df = checked_base_df.withColumn(  # Estimate the purchase cost of this order line.
    "estimated_line_cost", (F.col("quantity") * F.col("_product_cost_price")).cast("decimal(14,2)")  # Units times product unit cost.
).withColumn(  # Estimate margin before order-level discounts and tax.
    "estimated_line_margin", (F.col("line_total") - F.col("estimated_line_cost")).cast("decimal(14,2)")  # Line total minus estimated cost.
).drop("_product_cost_price")  # Keep only the two derived business fields.
print("Rows with derived cost and margin:", checked_base_df.count())  # Confirm the enriched row count.


In [ ]:
checked_df = checked_base_df.withColumn(  # Add duplicate-key counts.
    "id_occurrences", F.count(F.lit(1)).over(Window.partitionBy(*business_key))  # Count the cleaned key.
)
def issue_column(rules):  # Turn named validation rules into one issue string.
    return F.concat_ws("; ", *[  # Skip rules that do not match a row.
        F.when(F.coalesce(F.expr(predicate), F.lit(False)), F.lit(label))  # Evaluate each SQL predicate.
        for label, predicate in rules  # Use the rule names defined above.
    ]) if rules else F.lit("")  # Keep an empty string when no rules are defined.
checked_df = checked_df.withColumn("quality_issue", issue_column(rejection_rules))  # Reasons to reject.
checked_df = checked_df.withColumn("critical_issue", issue_column(critical_rules))  # Reasons to fail the build.
checked_df = checked_df.withColumn("warning_issue", issue_column(warning_rules))  # Nonblocking issues.
accepted_df = checked_df.filter(F.col("quality_issue") == "")  # Select clean rows.
rejected_df = checked_df.filter(F.col("quality_issue") != "")  # Select rejected rows.
accepted_count = accepted_df.count()  # Count Silver candidates.
rejected_count = rejected_df.count()  # Count error candidates.
critical_count = checked_df.filter(F.col("critical_issue") != "").count()  # Count critical failures.
warning_count = accepted_df.filter(F.col("warning_issue") != "").count()  # Count accepted warnings.
if accepted_count + rejected_count != bronze_count:  # Account for each Bronze row.
    raise ValueError("Validation lost source rows")  # Stop if counts differ.
print("Accepted:", accepted_count, "Rejected:", rejected_count)  # Show the split.
print("Critical:", critical_count, "Warnings:", warning_count)  # Show severities.


In [ ]:
run_id = str(uuid4())  # Give this run a unique audit label.
if rejected_count > 0:  # Save evidence when invalid rows exist.
    error_path = project_root / "error" / "data" / table_name  # Choose the ignored error folder.
    error_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure its parent exists.
    errors_df = (rejected_df.withColumn("run_id", F.lit(run_id))  # Identify this run.
                 .withColumn("rejected_at", F.current_timestamp()))  # Record processing time.
    errors_df.write.format("delta").mode("append").save(str(error_path))  # Keep earlier error runs.
    saved_errors_df = spark.read.format("delta").load(str(error_path))  # Read error evidence back.
    if saved_errors_df.filter(F.col("run_id") == run_id).count() != rejected_count:  # Verify this run.
        raise ValueError("Rejected rows were not saved correctly")  # Keep prior Silver intact.
if critical_count > 0:  # Apply workbook Critical / Fail pipeline rules.
    raise ValueError(f"{critical_count} critical failures; Silver was not overwritten")  # Stop safely.
silver_path = project_root / "lakehouse" / "silver" / "data" / table_name  # Choose the output path.
silver_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure the parent exists.
output_columns = expected_columns + ["estimated_line_cost", "estimated_line_margin", "warning_issue"]  # Keep derived cost and margin.
silver_df = accepted_df.select(*output_columns)  # Omit raw JSON and helper fields.
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(silver_path))  # Replace the snapshot and extend its schema.
saved_df = spark.read.format("delta").load(str(silver_path))  # Read the committed Delta version.
if saved_df.count() != accepted_count or saved_df.columns != output_columns:  # Reconcile saved rows.
    raise ValueError("Silver save did not match the accepted dataset")  # Report a mismatch.
print("Bronze:", bronze_count, "Silver:", saved_df.count(), "Rejected:", rejected_count)  # Show counts.
print("Warnings in Silver:", saved_df.filter(F.col("warning_issue") != "").count())  # Show warnings.
print("Delta log exists:", (silver_path / "_delta_log").is_dir())  # Confirm Delta output.


In [ ]:
saved_types = {f.name: f.dataType.simpleString() for f in saved_df.schema.fields}  # Inspect stored types.
for name, kind in fields:  # Check each typed business field.
    if kind not in ("text", "optional_text"):  # Text fields need no conversion check.
        expected_type = "int" if kind == "optional_int" else kind  # Normalize integer name.
        if saved_types[name] != expected_type:  # Compare actual and intended types.
            raise ValueError(f"{name}: expected {expected_type}, got {saved_types[name]}")  # Explain mismatch.
print("Saved types:", saved_types)  # Show the final Silver schema.
for derived_name in ("estimated_line_cost", "estimated_line_margin"):  # Verify both new measures.
    if saved_types[derived_name] != "decimal(14,2)":  # They must be exact two-decimal numbers.
        raise ValueError(f"Wrong stored type for {derived_name}")  # Stop on an unexpected schema.
print("Derived measures ready for Gold:", saved_df.select("estimated_line_cost", "estimated_line_margin").count())  # Confirm the row grain.
